In [74]:
# QUESTION = """
# Диагностика экономического и пространственного состояния г. Тюмень:
# - социально-экономический анализ (демография, производительность труда, структура и динамика ВГП, рынок труда, рынок жилья и коммерческой недвижимости, бюджетная обеспеченность);
# - пространственный анализ (функциональная организация территории, структура землевладения, природноэкологический каркас, архитектурно-градостроительная среда);
# - транспортный анализ (городской и внешний пассажирский и грузовой транспорт, парковки, пешеходные зоны);
# - анализ инженерной инфраструктуры (водоснабжение и водоотведение, энергоснабжение, теплоснабжение).
# """

QUESTION = """
Социально-экономический анализ г. Тюмень:
- демография, 
- производительность труда, 
- структура и динамика ВГП, 
- рынок труда, 
- рынок жилья и коммерческой недвижимости,
- бюджетная обеспеченность;
"""

## Board

In [75]:
from pydantic import BaseModel, Field, model_validator
from src.utils import get_id

class BaseNote(BaseModel):
    """Запись для добавления на доску"""
    content : str = Field(description='Содержимое записи')

class Note(BaseNote):
    id : str = Field(default='', description='ID записи')
    author_id : str = Field(description='ID автора записи')

    @model_validator(mode='after')
    def _validate_model(self):
        self.id = get_id()
        return self

In [76]:
class Board(BaseModel):
    notes : list[Note] = Field(default=[], description='Записи на доске')

    def add_note(self, base_note : BaseNote, author_id : str) -> str:
        note = Note(author_id=author_id, **base_note.model_dump())
        self.notes.append(note)
        return note.id
    
    def remove_notes(self, notes_ids : list[str]):
        self.notes = [n for n in self.notes if n.id not in notes_ids]

    def to_str(self):
        return 'Содержимое доски:\n' + str(self.notes)

In [77]:
class BlackBoard():

    def __init__(self, question : str):
        self.question = question
        self.board = Board()

    def run(self, iterations : int = 3):
        ...

## Agents

In [78]:
from src import Agent
from langchain.messages import SystemMessage

WORKER_PROMPT = """
Ваш личный ID {id}.
Вы {role_name}, сотрудничающий с другими агентами для решения задачи.
Существует общая доска, которую каждый из вас может читать и на которой может оставлять записи.
---
Описание роли:
{role_description}
---
Задача:
{question}
---
"""

def create_worker_agent(
    system_prompt, 
    role_name : str, 
    role_description : str, 
    tools : list | None = None, 
    response_format : type[BaseModel] | None = None
) -> Agent:
    return Agent(
        WORKER_PROMPT + '\n' + system_prompt, 
        tools=tools, 
        response_format=response_format,
        metadata={
            'role_name': role_name,
            'role_description': role_description,
            'question': QUESTION
        }
    )

### Cleaner

In [79]:
CLEANER_PROMPT = """
Проанализируйте записи на общей доске и определите любые бесполезные или избыточные.

Если вы обнаружите такие записи:
- Перечислите их

Если бесполезных записей нет, верните пустой список.
"""

class CleanerResponse(BaseModel):
    notes_ids : list[str] = Field(default=[], description='Список ID записей к удалению')

cleaner_agent = create_worker_agent(
    CLEANER_PROMPT,
    role_name='Уборщик',
    role_description='Анализирует доску, выявляет и удаляет бесполезные или избыточные записи',
    response_format=CleanerResponse
)

### Critic

In [ ]:
CRITIC_PROMPT = """
Проанализируйте записи на общей доске и определите любые бесполезные или избыточные.

Если вы обнаружите такие записи, опишите их и объясните, почему они должны быть удалены.
Если бесполезных записей нет, просто укажите, что бесполезных записей нет и вы ожидаете дополнительной информации.
"""

critic_agent = create_worker_agent(
    CRITIC_PROMPT,
    role_name='Критик',
    role_description='Выявляет ошибочные или вводящие в заблуждение записи на доске',
    response_format=BaseNote
)

### Decider

In [81]:
DECIDER_PROMPT = """
Ваша задача проанализировать текущее состояние общей доски и решить, достаточно ли у команды информации для получения окончательного ответа.
Если информации на доске достаточно для решения задачи, вы должны указать, что работа завершена. 
Если для получения ответа необходима дополнительная информация от других агентов, вы должны указать, что процесс следует продолжить.
Не решайте задачу.
"""

class DeciderResponse(BaseModel):
    note : BaseNote = Field(description='Запись для добавления на доску')
    is_final : bool = Field(default=False, description='Сигнал о завершении процесса работы над задачей')

decider_agent = create_worker_agent(
    DECIDER_PROMPT,
    role_name='Арбитр',
    role_description='Оценивает полноту информации. Останавливает либо инициирует продолжение обсуждения',
    response_format=DeciderResponse
)

### Planner

In [82]:
PLANNER_PROMPT = """
Ваша задача проанализировать текущее состояние общей доски и решить, достаточно ли у команды информации для получения окончательного ответа.
Если информации на доске достаточно для решения задачи, вы должны указать, что работа завершена. 
Если для получения ответа необходима дополнительная информация от других агентов, вы должны указать, что процесс следует продолжить.
Не решайте задачу.
"""

planner_agent = create_worker_agent(
    PLANNER_PROMPT,
    role_name='Планировщик',
    role_description='Разрабатывает пошаговый план решения задачи на основе содержимого доски',
    response_format=BaseNote
)

### Experts

In [93]:
from src.tools import ddgs_tool

WEB_PROMPT = """
Основываясь на результатах вызова инструментов и текущем содержимом общей доски, решите задачу, изложите свои идеи и информацию, которую вы хотите записать на доску.
Совершенно не обязательно полностью соглашаться с точкой зрения, представленной на доске.
НЕ ИСПОЛЬЗУЙТЕ общие знания.
"""

web_agent = create_worker_agent(
    WEB_PROMPT,
    'Эксперт по поиску в интернете',
    'Ищет информацию в интернете',
    tools=[ddgs_tool],
    response_format=BaseNote
)

In [94]:
experts = [web_agent]

In [95]:
workers = {w.id: w for w in [
    planner_agent, *experts, critic_agent, cleaner_agent, decider_agent
]}

### Controller

In [96]:
CONTROLLER_PROMPT = """
Ваша задача назначить других агентов для сотрудничества и решения данной задачи.
Агенты обмениваются информацией через общую доску.
Основываясь на содержимом, которое уже есть на доске, вам необходимо выбрать подходящих агентов из списка, чтобы они оставили записи на доске.
---
Перечень доступных агентов:
{workers}
---
Данная задача:
{question}
"""

class ControllerResponse(BaseModel):
    agents_ids : list[str] = Field(min_length=1, description='Упорядоченный список ID агентов')

controller_agent = Agent(
    system_prompt=CONTROLLER_PROMPT,
    metadata={
        'workers': [{
            'id': w.id,
            'role_name': w.metadata['role_name'],
            'role_description': w.metadata['role_description']
        } for w in workers.values()],
        'question': QUESTION,
    },
    response_format=ControllerResponse
)

## Expertiment

In [97]:
worker.metadata

{'id': 'fa8885',
 'role_name': 'Эксперт по поиску в интернете',
 'role_description': 'Ищет информацию в интернете',
 'question': '\nСоциально-экономический анализ г. Тюмень:\n- демография, \n- производительность труда, \n- структура и динамика ВГП, \n- рынок труда, \n- рынок жилья и коммерческой недвижимости,\n- бюджетная обеспеченность;\n'}

In [98]:
from tqdm import tqdm

board = Board()

is_final = False

for i in range(3):
    agents_ids = controller_agent.invoke(board.to_str()).agents_ids
    for agent_id in tqdm(agents_ids):
        worker = workers[agent_id]
        response = worker.invoke(board.to_str())
        
        if worker == decider_agent:
            note = response.note
            is_final = response.is_final
        elif worker == cleaner_agent:
            notes_ids = response.notes_ids
            board.remove_notes(notes_ids)
            continue
        else:
            note = response
        
        board.add_note(note, worker.id)

        if is_final:
            break

    if is_final:
        break

  0%|          | 0/5 [00:06<?, ?it/s]


KeyboardInterrupt: 

In [104]:
print(board.notes[0].content)

Собранные данные по Тюмени (2023 г.)

**1. Демография**
- Численность постоянного населения города Тюмень — 855 618 человек (данные на 1 января 2023 г.)【5†L1-L3】.
- По данным на 1 января 2024 г. население составило 861 098 человек, что позволяет оценить рост в ≈ 0,6 % за год【6†L1-L3】.
- Прирост населения в Тюменской области в 2023 году составил 1 245 человек на 1000 человек (13‑е место в России, 3‑е в УФО)【6†L1-L2】 – полезно для оценки миграционных потоков в город.

**2. Производительность труда**
- Поиск не дал конкретных региональных показателей (выработка, трудоёмкость, индекс ПТ) для Тюмени в 2023 году. На данный момент доступны лишь общие материалы о проектах повышения ПТ в Тюменской области, но без количественных данных【2†L1-L4】.

**3. Структура и динамика ВГП**
- Поиск не выявил официальных цифр ВГП города Тюмень или его отраслевой структуры за 2023 год. Есть упоминание о аналитическом отчёте Института экономики города, но ссылка не содержит открытых цифр【3†L1-L4】.

**4. Рынок т

In [108]:
agent = Agent(system_prompt=f"""
На основе содержимого доски сформулируй финальный ответ на задачу.
Правила работы:
- Только фактический ответ по задаче
- Без вступлений, выводов, рекомендаций и пояснений
- Не добавлять ничего сверх найденной информации
- Кратко и по существу
---
Данная задача:
{QUESTION}
""")

res = agent.invoke(SystemMessage(board.to_str()))

In [109]:
print(res)

**Демография**  
- Численность постоянного населения города Тюмень — 855 618 человек (на 1 января 2023 г.)【5†L1-L3】.  
- Численность населения — 861 098 человек (на 1 января 2024 г.), рост ≈ 0,6 % за год【6†L1-L3】.  
- Прирост населения в Тюменской области в 2023 году — 1 245 человек на 1000 человек (13‑е место в России, 3‑е в УФО)【6†L1-L2】.  

**Производительность труда** – данные за 2023 год не найдены.  

**Структура и динамика ВГП** – официальные цифры за 2023 год не найдены.  

**Рынок труда** – региональные показатели за 2023 год не найдены.  

**Рынок жилья и коммерческой недвижимости** – конкретные данные за 2023 год не найдены.  

**Бюджетная обеспеченность** – сведения о доходах и расходах бюджета за 2023 год не найдены.  
